In [1]:
%load_ext autoreload
%autoreload 2

# How to use it

In [ ]:
from typing import Any

import boto3

from brollm import BaseContract
from brollm.retry import retry
from brollm.parsing import extract_codeblocks
from dataclasses import dataclass

@dataclass
class Tokens:
    input: int
    output: int

@dataclass
class ResponseModel:
    content: str
    usage: Tokens

# you can design user_message that fits your model providers
def user_message(text: str, image_bytes: bytes | None = None, image_format: str | None = None) -> dict[str, Any]:
    content: list[dict[str, Any]] = [{"text": text}]
    if image_bytes:
        content.append({"image": {"format": image_format, "source": {"bytes": image_bytes}}})
    return {"role": "user", "content": content}

# you can design ai_message that fits your model providers
def ai_message(text: str) -> dict[str, Any]:
    return {"role": "assistant", "content": [{"text": text}]}

# this can be a function that return a class, or you can create input_fn and output_fn as its own outside this wrapper.
def make_bedrock_chat(
        model_name: str = "google.gemma-3-4b-it",
        temperature: float = 0,
        region_name: str = "us-west-2",
        aws_access_key_id: str | None = None,
        aws_secret_access_key: str | None = None,
        aws_session_token: str | None = None,
        max_attempts: int = 3,
) -> BaseContract:
    client = boto3.client(
        service_name="bedrock-runtime",
        region_name=region_name,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token,
    )

    # this decorator is optional, but it will be a quick way to create a retry mechanism
    # you can discard or have it your way baked in the input_fn directly
    @retry(max_attempts=max_attempts, on=(client.exceptions.ThrottlingException,))
    def input_fn(system_prompt: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
        return client.converse(
            modelId=model_name,
            messages=messages,
            system=[{"text": system_prompt}],
            inferenceConfig={"temperature": temperature},
        )

    # here you can return your response as a schema or just plain text, it's upto you
    # you can also parse it as a json or any other format you want in here
    def output_fn(response: dict[str, Any]) -> ResponseModel:
        return ResponseModel(
            content=response["output"]["message"]["content"][0]["text"],
            usage=Tokens(
                input=response["usage"]["inputTokens"],
                output=response["usage"]["outputTokens"],
            ),
        )        

    return BaseContract(input_fn=input_fn, output_fn=output_fn)


In [3]:
model = make_bedrock_chat(model_name="google.gemma-3-4b-it", region_name='us-east-1')

In [4]:
system_prompt = "You're a helpful assistant."
messages = [user_message(text='Hi')]
response = model(system_prompt, messages)
response

ResponseModel(content='Hi there! How can I help you today? 😊 \n\nDo you want to:\n\n*   Ask me a question?\n*   Brainstorm ideas?\n*   Get help with a task?\n*   Just chat?\n\nLet me know what’s on your mind!', usage=Tokens(input=18, output=60))

In [5]:
print(response.content)

Hi there! How can I help you today? 😊 

Do you want to:

*   Ask me a question?
*   Brainstorm ideas?
*   Get help with a task?
*   Just chat?

Let me know what’s on your mind!


In [6]:
system_prompt = """\
You are a senior {language} developer. You can do everything in {language}. 

Cautions:
- Return only {language} codeblock. Discard irrelevant conversation.

Always return your response in {language} codeblock like below example:
```{language}
<your code>
```
""".strip()

messages = [
    user_message(text="Write me a code for getting current datetime.")
]

response = model.run(system_prompt=system_prompt.format(language="python"), messages=messages)
response

ResponseModel(content='```python\nimport datetime\n\nnow = datetime.datetime.now()\nprint(now)\n```', usage=Tokens(input=74, output=22))

In [7]:
codes = []
for cb in extract_codeblocks(response.content):
    codes.append(cb.content)
    print(cb.content)

import datetime

now = datetime.datetime.now()
print(now)



In [8]:
# it returns corerctly since it's just a normal syntax
for c in codes:
    exec(c)

2026-09-08 23:51:12.254338


In [9]:
messages = [
    user_message(text="I wanna have a dataclass for customer information on id, first and last name, age, gender and other important information.")
]

response = model.run(system_prompt=system_prompt.format(language="python"), messages=messages)
print(response)

ResponseModel(content='```python\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\n@dataclass\nclass Customer:\n    id: int\n    first_name: str\n    last_name: str\n    age: int\n    gender: str\n    email: Optional[str] = None\n    phone_number: Optional[str] = None\n    address: Optional[str] = None\n    loyalty_points: int = 0\n    membership_since: Optional[str] = None  # e.g., "2023-01-15"\n\nif __name__ == \'__main__\':\n    # Example Usage\n    customer1 = Customer(\n        id=12345,\n        first_name="Alice",\n        last_name="Smith",\n        age=30,\n        gender="Female",\n        email="alice.smith@example.com",\n        phone_number="555-123-4567",\n        address="123 Main St, Anytown",\n        loyalty_points=150,\n        membership_since="2022-05-20"\n    )\n\n    customer2 = Customer(\n        id=67890,\n        first_name="Bob",\n        last_name="Johnson",\n        age=25,\n        gender="Male"\n    )\n\n    print(customer1)\n    p

In [10]:
codes = []
for cb in extract_codeblocks(response.content):
    codes.append(cb.content)
    print(cb.content)

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Customer:
    id: int
    first_name: str
    last_name: str
    age: int
    gender: str
    email: Optional[str] = None
    phone_number: Optional[str] = None
    address: Optional[str] = None
    loyalty_points: int = 0
    membership_since: Optional[str] = None  # e.g., "2023-01-15"

if __name__ == '__main__':
    # Example Usage
    customer1 = Customer(
        id=12345,
        first_name="Alice",
        last_name="Smith",
        age=30,
        gender="Female",
        email="alice.smith@example.com",
        phone_number="555-123-4567",
        address="123 Main St, Anytown",
        loyalty_points=150,
        membership_since="2022-05-20"
    )

    customer2 = Customer(
        id=67890,
        first_name="Bob",
        last_name="Johnson",
        age=25,
        gender="Male"
    )

    print(customer1)
    print(customer2)



In [11]:
# it returns correctly since the codeblock here is just an execution file.
for c in codes:
    exec(c)

Customer(id=12345, first_name='Alice', last_name='Smith', age=30, gender='Female', email='alice.smith@example.com', phone_number='555-123-4567', address='123 Main St, Anytown', loyalty_points=150, membership_since='2022-05-20')
Customer(id=67890, first_name='Bob', last_name='Johnson', age=25, gender='Male', email=None, phone_number=None, address=None, loyalty_points=0, membership_since=None)


In [12]:
system_prompt = """\
You are a senior python developer. Your job is to help user write a python dataclass based on the specification. 

Cautions:
- Return only python codeblock. Discard irrelevant conversation.

Always return your response in python codeblock like below example:
```python
# your dataclass
<your code>
```
""".strip()

messages = [
    user_message(text="I wanna have a dataclass for customer information on id, first and last name, age, gender and other important information.")
]

response = model.run(system_prompt=system_prompt, messages=messages)
print(response)

ResponseModel(content='```python\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\n@dataclass\nclass Customer:\n    id: int\n    first_name: str\n    last_name: str\n    age: int\n    gender: str\n    email: Optional[str] = None\n    phone_number: Optional[str] = None\n    address: Optional[str] = None\n    loyalty_points: int = 0\n    membership_since: Optional[str] = None\n```', usage=Tokens(input=104, output=111))


In [13]:
codes = []
for cb in extract_codeblocks(response.content):
    codes.append(cb.content)
    print(cb.content)

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Customer:
    id: int
    first_name: str
    last_name: str
    age: int
    gender: str
    email: Optional[str] = None
    phone_number: Optional[str] = None
    address: Optional[str] = None
    loyalty_points: int = 0
    membership_since: Optional[str] = None



In [14]:
# return nothing is correct since it's just a dataclass design, not real implementation.
for c in codes:
    exec(c)